# Joshi Part 7: Exotic Engine - Path-Dependent Options (Rust)

Rust kernel version of `08_joshi_path_dependent.ipynb`.

## Setup

In [ ]:
:dep RustQuant = { path = "../crates/RustQuant" }
:dep time = { version = "0.3", features = ["macros"] }

In [ ]:
use time::macros::date;
use RustQuant::instruments::options::*;
use RustQuant::instruments::*;
use RustQuant::stochastics::*;

let spot = 100.0;
let strike = 100.0;
let rate = 0.05;
let vol = 0.20;
let expiry = date!(2027 - 03 - 22);

let gbm = GeometricBrownianMotion::new(rate, vol);
let config = StochasticProcessConfig::new(
    spot, 0.0, 1.0, 252, StochasticScheme::EulerMaruyama, 100_000, true, None,
);

let contract_call = OptionContractBuilder::default()
    .type_flag(TypeFlag::Call).exercise_flag(ExerciseFlag::European { expiry })
    .strike_flag(Some(StrikeFlag::Fixed)).build().unwrap();
let contract_put = OptionContractBuilder::default()
    .type_flag(TypeFlag::Put).exercise_flag(ExerciseFlag::European { expiry })
    .strike_flag(Some(StrikeFlag::Fixed)).build().unwrap();

## Reference: Vanilla European

In [ ]:
let v_call = EuropeanVanillaOption::new(strike, expiry, TypeFlag::Call)
    .price_monte_carlo(&gbm, &config, rate);
let v_put = EuropeanVanillaOption::new(strike, expiry, TypeFlag::Put)
    .price_monte_carlo(&gbm, &config, rate);
println!("Vanilla Call: {:.4}", v_call);
println!("Vanilla Put:  {:.4}", v_put);

## 1. Asian Options

Payoff depends on the **average** price. Cheaper than vanilla due to averaging.

In [ ]:
let asian_arith = AsianOption::new(
    contract_call.clone(), AveragingMethod::ArithmeticDiscrete, Some(strike),
);
let asian_geo = AsianOption::new(
    contract_call.clone(), AveragingMethod::GeometricDiscrete, Some(strike),
);
let float_contract = OptionContractBuilder::default()
    .type_flag(TypeFlag::Call).exercise_flag(ExerciseFlag::European { expiry })
    .strike_flag(Some(StrikeFlag::Floating)).build().unwrap();
let asian_float = AsianOption::new(float_contract, AveragingMethod::ArithmeticDiscrete, None);

println!("Arithmetic Asian Call: {:.4} (vanilla: {:.4})",
    asian_arith.price_monte_carlo(&gbm, &config, rate), v_call);
println!("Geometric Asian Call:  {:.4}",
    asian_geo.price_monte_carlo(&gbm, &config, rate));
println!("Floating Strike Asian: {:.4}",
    asian_float.price_monte_carlo(&gbm, &config, rate));

## 2. Barrier Options

Knock-In + Knock-Out = Vanilla (parity).

In [ ]:
let uo = BarrierOption::new(contract_call.clone(), BarrierType::UpAndOut, 120.0, strike);
let ui = BarrierOption::new(contract_call.clone(), BarrierType::UpAndIn, 120.0, strike);
let do_ = BarrierOption::new(contract_call.clone(), BarrierType::DownAndOut, 80.0, strike);
let di = BarrierOption::new(contract_call.clone(), BarrierType::DownAndIn, 80.0, strike);

let uo_p = uo.price_monte_carlo(&gbm, &config, rate);
let ui_p = ui.price_monte_carlo(&gbm, &config, rate);
let do_p = do_.price_monte_carlo(&gbm, &config, rate);
let di_p = di.price_monte_carlo(&gbm, &config, rate);

println!("Up Barrier=120:");
println!("  Up-Out: {:.4}, Up-In: {:.4}, Sum: {:.4} (vanilla: {:.4})", uo_p, ui_p, uo_p+ui_p, v_call);
println!("Down Barrier=80:");
println!("  Down-Out: {:.4}, Down-In: {:.4}, Sum: {:.4} (vanilla: {:.4})", do_p, di_p, do_p+di_p, v_call);

## 3. Lookback Options

Most expensive — payoff based on path extremum.

In [ ]:
let lb_fixed_call = LookbackOption::new(contract_call.clone(), Some(strike));
let lb_fixed_put = LookbackOption::new(contract_put.clone(), Some(strike));

let fc = OptionContractBuilder::default()
    .type_flag(TypeFlag::Call).exercise_flag(ExerciseFlag::European { expiry })
    .strike_flag(Some(StrikeFlag::Floating)).build().unwrap();
let fp = OptionContractBuilder::default()
    .type_flag(TypeFlag::Put).exercise_flag(ExerciseFlag::European { expiry })
    .strike_flag(Some(StrikeFlag::Floating)).build().unwrap();
let lb_float_call = LookbackOption::new(fc, None);
let lb_float_put = LookbackOption::new(fp, None);

println!("Fixed Strike:");
println!("  Lookback Call: {:.4} (vanilla: {:.4})",
    lb_fixed_call.price_monte_carlo(&gbm, &config, rate), v_call);
println!("  Lookback Put:  {:.4} (vanilla: {:.4})",
    lb_fixed_put.price_monte_carlo(&gbm, &config, rate), v_put);
println!("Floating Strike:");
println!("  Call (S_T - S_min): {:.4}", lb_float_call.price_monte_carlo(&gbm, &config, rate));
println!("  Put  (S_max - S_T): {:.4}", lb_float_put.price_monte_carlo(&gbm, &config, rate));

## Price Ordering

Asian (cheapest) < Vanilla < Lookback (most expensive)

Barrier In + Barrier Out = Vanilla (parity)